In [1]:
from transformers import BertTokenizer, BertForSequenceClassification
import torch

model_name  = 'klue/bert-base'
# 한국어 토크나이져
tokenizer = BertTokenizer.from_pretrained(model_name)
# 분류기
model = BertForSequenceClassification.from_pretrained(model_name, num_labels = 2)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: klue/bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(32000, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [ ]:
from torch.optim import AdamW
from tqdm import tqdm
# 소규모 훈련 데이터 구축 및 학습(Fine-tuning)
train_texts = [
    "이 영화 진짜 개꿀잼이네요 ㅋㅋㅋ 완전 강추!",   # 긍정
    "배우들 연기가 훌륭하고 스토리가 감동적입니다.", # 긍정
    "돈 아깝네요 최악입니다. 절대 보지 마세요.",     # 부정
    "스토리가 너무 지루하고 결말이 어이없음.",       # 부정
]
train_labels = [1, 1, 0, 0] # 1: 긍정, 0: 부정
# 토크나이징
inputs = tokenizer(train_texts,padding=True, truncation=True,return_tensors='pt').to(device)
labels = torch.tensor(train_labels).to(device)
optimizer =AdamW(model.parameters(), lr = 5e-5)
model.train()
epochs=3

# 허깅페이스에 올라간 모델들은 loss함수를 내장하고 있어서 훈류인지 회귀인지 알아서 알맞은 손실함수를 자동 적용
# 입력과 정답을 주면 알아서....

for epoch in tqdm(range(epochs)):
    optimizer.zero_grad()
    outputs = model(**inputs,labels=labels)
    loss = outputs.loss
    loss.backward()
    optimizer.step()
    print(f'epoch: {epoch+1} /{epochs}  loss : {loss.item()}')


 33%|███▎      | 1/3 [00:02<00:05,  2.74s/it]

epoch: 1 /3  loss : 0.7323769330978394


 67%|██████▋   | 2/3 [00:03<00:01,  1.81s/it]

epoch: 2 /3  loss : 0.45677512884140015


100%|██████████| 3/3 [00:05<00:00,  1.72s/it]

epoch: 3 /3  loss : 0.21956036984920502


In [9]:
# 추론
model.eval()
test_texts = [
    "어휴.. 진자 돈 아깝네. 보지마삼",
    "이 영화 어이없음",
    "시간 가는 줄 모르고 봤습니다. 인생작 등극!",
    "도대체 무슨 내용을 말하고 싶은 건지 모르겠네요. 돈버림."
]

test_inputs = tokenizer(test_texts,padding=True, truncation=True, return_tensors='pt').to(device)
with torch.no_grad():
    outputs = model(**test_inputs)
    logits = outputs.logits   
    predictions = torch.argmax(logits,dim=-1)
    print(predictions)

tensor([0, 0, 1, 0])
